# Centered Kernel Alignment (CKA) for Genomic Language Model Embeddings
Notebook for computing batched linear CKA across gLM embedding CSVs.

Designed for sequence‑pooled embeddings with identical ordering across models.

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
from itertools import combinations
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations, product

## Linear CKA Implementation (feature-space, memory efficient)

In [ ]:
def center_features(X):
    return X - X.mean(axis=0, keepdims=True)

def linear_cka(X, Y):
    X = center_features(X)
    Y = center_features(Y)

    XT_Y = X.T @ Y
    HSIC = np.linalg.norm(XT_Y, "fro") ** 2

    XT_X = X.T @ X
    YT_Y = Y.T @ Y

    norm_x = np.linalg.norm(XT_X, "fro")
    norm_y = np.linalg.norm(YT_Y, "fro")

    return HSIC / (norm_x * norm_y + 1e-12)

## Load Embeddings

In [ ]:
DATA_DIR = "GB_sequence_embeddings"

model_files = [
    "gena-lm-bigbird-base-t2t_embeddings.csv",
    "gpn-brassiscales_embeddings.csv",
    "GROVER_embeddings.csv",
    "hyenadna-large-1m-seqlen-hf_embeddings.csv",
    "hyenadna-medium-450k-seqlen-hf_embeddings.csv",
    "nucleotide-transformer-2.5b-1000g_embeddings.csv",
    "nucleotide-transformer-2.5b-multi_species_embeddings.csv",
    "nucleotide-transformer-500m-human-ref_embeddings.csv",
]

def load_embeddings(path):
    print(f"Loading {path}")
    df = pd.read_csv(path)

    embeddings = np.vstack(
        df["embedding"].apply(lambda x: np.array(ast.literal_eval(x))).values
    )
    labels = df["label"].values
    return embeddings, labels

model_embeddings = {}
model_labels = None

for f in model_files:
    name = f.replace("_embeddings.csv", "")
    emb, labels = load_embeddings(os.path.join(DATA_DIR, f))
    model_embeddings[name] = emb

    if model_labels is None:
        model_labels = labels

## Sanity Checks

In [ ]:
for k, v in model_embeddings.items():
    print(k, v.shape)

assert len(model_labels) == list(model_embeddings.values())[0].shape[0]
print("Sanity checks passed.")

## Batched CKA Computation

In [ ]:
def batched_cka(X, Y, batch_size=2000):
    n = X.shape[0]
    scores = []

    for start in tqdm(range(0, n, batch_size)):
        end = min(start + batch_size, n)

        Xb = X[start:end]
        Yb = Y[start:end]

        if len(Xb) < 2:
            continue

        score = linear_cka(Xb, Yb)
        scores.append(score)

    return float(np.mean(scores))

## Compute Pairwise Model Alignment

In [ ]:
results = []
model_names = list(model_embeddings.keys())

for m1, m2 in combinations(model_names, 2):
    print(f"Computing CKA: {m1} vs {m2}")
    X = model_embeddings[m1]
    Y = model_embeddings[m2]

    score = batched_cka(X, Y, batch_size=2000)

    results.append({
        "model_1": m1,
        "model_2": m2,
        "cka_score": score
    })

results_df = pd.DataFrame(results).sort_values("cka_score", ascending=False)
results_df

In [ ]:
def batched_cka_subset(X, Y, labels, target_label, batch_size=2000):
    idx = np.where(labels == target_label)[0]

    X_sub = X[idx]
    Y_sub = Y[idx]

    return batched_cka(X_sub, Y_sub, batch_size=batch_size)

In [ ]:
def subset_cka(X, Y, labels, label_X, label_Y, batch_size=2000):
    idx_X = np.where(labels == label_X)[0]
    idx_Y = np.where(labels == label_Y)[0]

    # Intersection of positions (since order is identical across models)
    # We only want positions where both models use the same sequences
    # Since ordering is identical, we just use idx_X ∩ idx_Y
    idx = np.intersect1d(idx_X, idx_Y)

    X_sub = X[idx]
    Y_sub = Y[idx]

    return batched_cka(X_sub, Y_sub, batch_size=batch_size)

In [ ]:
regimes = {
    (1, 1): "coding_vs_coding",
    (0, 0): "noncoding_vs_noncoding",
    (1, 0): "coding_vs_noncoding",
    (0, 1): "noncoding_vs_coding",
}

results = []

model_names = list(model_embeddings.keys())

for m1, m2 in combinations(model_names, 2):
    print(f"\n=== {m1} vs {m2} ===")

    X = model_embeddings[m1]
    Y = model_embeddings[m2]

    for (label_X, label_Y), regime_name in regimes.items():
        print(f"Computing {regime_name}...")

        score = subset_cka(
            X, Y, model_labels,
            label_X=label_X,
            label_Y=label_Y,
            batch_size=2000
        )

        results.append({
            "model_1": m1,
            "model_2": m2,
            "regime": regime_name,
            "cka_score": score
        })

results_df = pd.DataFrame(results)

results_df.to_csv("cka_full_regime_results.csv", index=False)

print("Saved: cka_full_regime_results.csv")

ranked_df = results_df.sort_values("cka_score", ascending=False)
ranked_df.to_csv("cka_ranked_results.csv", index=False)

print("Saved: cka_ranked_results.csv")

display(ranked_df.head(20))

In [ ]:
pivot_df = results_df.pivot_table(
    index=["model_1", "model_2"],
    columns="regime",
    values="cka_score"
).reset_index()

pivot_df.to_csv("cka_regime_pivot_table.csv", index=False)

print("Saved: cka_regime_pivot_table.csv")

display(pivot_df.head())


heatmap_df = results_df.pivot_table(
    index="model_1",
    columns="model_2",
    values="cka_score",
    aggfunc="mean"
)

plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="viridis")
plt.title("Average CKA Across All Regimes")
plt.tight_layout()
plt.savefig("cka_average_heatmap.png", dpi=300)
plt.show()

print("Saved: cka_average_heatmap.png")